# Q1 targeted notebook: plots, comparisons, and tables

This notebook is designed for **Question 1** and follows the uploaded project code and algorithm.

It includes the following tasks:

1. Plot the **temperature data scatter** and **piecewise linear interpolation** for the first 1800 s.
2. Plot the **moisture data scatter** and **piecewise linear interpolation** for the first 1800 s.
3. Generate a table comparing **Linear / PCHIP / MAKIMA** interpolation in the subsequent PDE solution.
4. Generate **result1.xlsx** using the Q1 algorithm with **N = 1280**, and export **Table 1** and **Table 2** as CSV.
5. Plot **temperature** and **moisture** heatmaps versus **time** and **radius**.
6. Plot profiles of temperature and moisture versus radius at  
   **100, 300, 600, 900, 1200, 1500, 1800 s**.
7. Compare **our FVM algorithm** with a **finite difference method (FDM)** using error tables and figures.

All figures are saved in **PNG** and **SVG** formats.  
All tables are saved in **CSV** format, with an additional Excel summary workbook.


In [2]:
from pathlib import Path
import sys
import json
import zipfile
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import PchipInterpolator, Akima1DInterpolator


# ============================================================
# 0. Paths
# ============================================================
ROOT = Path.cwd()

BUNDLE = ROOT / "q1_drying_project_bundle.zip"
EXTRACT_DIR = ROOT / "q1_project_extracted"

OUT = ROOT / "Q1_targeted_outputs"
FIG = OUT / "figures"
TAB = OUT / "tables"
NUM = OUT / "numerical_results"

for p in [OUT, FIG, TAB, NUM]:
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. Extract the uploaded Q1 project package
# ============================================================
solver_path = EXTRACT_DIR / "src" / "q1_solver.py"

if not solver_path.exists():
    if not BUNDLE.exists():
        raise FileNotFoundError(
            f"Cannot find project bundle:\n{BUNDLE}\n\n"
            "Please place q1_drying_project_bundle.zip "
            "in the same directory as this notebook."
        )

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(BUNDLE, "r") as z:
        z.extractall(EXTRACT_DIR)


# ============================================================
# 2. Dynamically load the original Q1 solver
# ============================================================
solver_path = EXTRACT_DIR / "src" / "q1_solver.py"

if not solver_path.exists():
    raise FileNotFoundError(
        f"Cannot find q1_solver.py after extraction:\n{solver_path}"
    )

module_name = "q1_solver"

# Important:
# dataclasses + `from __future__ import annotations`
# require the module to exist in sys.modules during execution.
spec = importlib.util.spec_from_file_location(
    module_name,
    solver_path
)

if spec is None or spec.loader is None:
    raise ImportError(
        f"Failed to create import specification for:\n{solver_path}"
    )

q1 = importlib.util.module_from_spec(spec)

# ---- Critical fix for Python 3.12 / dataclasses ----
sys.modules[module_name] = q1

try:
    spec.loader.exec_module(q1)
except Exception:
    # Avoid leaving a partially imported module behind
    sys.modules.pop(module_name, None)
    raise


# ============================================================
# 3. Environment data path
# ============================================================
ENV_PATH = EXTRACT_DIR / "data" / "附件1.xlsx"

if not ENV_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find 附件1.xlsx:\n{ENV_PATH}"
    )


# ============================================================
# 4. Quick verification
# ============================================================
print("=" * 60)
print("Q1 project loaded successfully")
print("=" * 60)

print(f"Solver file : {solver_path}")
print(f"Data file   : {ENV_PATH}")
print(f"Output dir  : {OUT}")

print("\nAvailable core functions:")
for name in [
    "Q1Parameters",
    "SolverConfig",
    "EnvironmentData",
    "solve_q1",
    "sample_solution",
    "paper_tables",
    "result_grid",
    "write_result1",
]:
    print(
        f"  {name:<24}",
        "OK" if hasattr(q1, name) else "MISSING"
    )


Q1 project loaded successfully
Solver file : /fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/q1_project_extracted/src/q1_solver.py
Data file   : /fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/q1_project_extracted/data/附件1.xlsx
Output dir  : /fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs

Available core functions:
  Q1Parameters             OK
  SolverConfig             OK
  EnvironmentData          OK
  solve_q1                 OK
  sample_solution          OK
  paper_tables             OK
  result_grid              OK
  write_result1            OK


In [3]:

# ------------------------------------------------------------------
# Plotting style: clean, compact, paper-friendly
# ------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 8.5,
    "axes.labelsize": 9,
    "axes.titlesize": 9.5,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.6,
    "lines.markersize": 4,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.margins(x=0.02)

def save_dual(fig, stem):
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight", dpi=300)
    fig.savefig(FIG / f"{stem}.svg", bbox_inches="tight")
    plt.close(fig)

def save_df(df, stem):
    csv_path = TAB / f"{stem}.csv"
    df.to_csv(csv_path, index=True if df.index.name is not None else False, encoding="utf-8-sig")
    return csv_path


## 1. Read the ambient data for the first 1800 s

In [4]:

env = q1.EnvironmentData.from_excel(ENV_PATH)
p = q1.Q1Parameters()
cfg = q1.SolverConfig(method="BDF", rtol=1e-9, atol=1e-11, max_step=1.0)

env_df = pd.read_excel(ENV_PATH)
env_df = env_df[["时间", "温度", "水分浓度"]].copy()
env_df = env_df[(env_df["时间"] >= 0) & (env_df["时间"] <= 1800)].sort_values("时间").reset_index(drop=True)
env_df.columns = ["time_s", "temperature_C", "moisture_kgkg"]

t_env = env_df["time_s"].to_numpy(dtype=float)
T_env = env_df["temperature_C"].to_numpy(dtype=float)
C_env = env_df["moisture_kgkg"].to_numpy(dtype=float)

env_df.head()


,time_s,temperature_C,moisture_kgkg
0,0,28.000,0.01963
1,60,28.528,0.02002
2,120,29.192,0.02053
3,180,29.684,0.02100
4,240,30.037,0.02151


## 2. Plot the temperature scatter and piecewise linear interpolation

In [5]:

dense_t = np.linspace(0, 1800, 1801)
T_linear = np.interp(dense_t, t_env, T_env)

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.scatter(t_env, T_env, label="Data points")
ax.plot(dense_t, T_linear, label="Piecewise linear interpolation")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Ambient Temperature (°C)")
ax.set_title("Ambient Temperature in the First 1800 s")
clean_axis(ax)
ax.legend(frameon=False)
save_dual(fig, "Fig1_temperature_scatter_linear")

print("Saved: Fig1_temperature_scatter_linear")


Saved: Fig1_temperature_scatter_linear


## 3. Plot the moisture scatter and piecewise linear interpolation

In [6]:

C_linear = np.interp(dense_t, t_env, C_env)

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.scatter(t_env, C_env, label="Data points")
ax.plot(dense_t, C_linear, label="Piecewise linear interpolation")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Ambient Moisture Concentration (kg/kg)")
ax.set_title("Ambient Moisture Concentration in the First 1800 s")
clean_axis(ax)
ax.legend(frameon=False)
save_dual(fig, "Fig2_moisture_scatter_linear")

print("Saved: Fig2_moisture_scatter_linear")


Saved: Fig2_moisture_scatter_linear


## 4. Define interpolation helpers and a custom FVM/FDM comparison framework

In [7]:

def build_interpolator(name, x, y):
    name = name.lower()
    if name == "linear":
        return lambda q: np.interp(np.asarray(q, dtype=float), x, y)
    if name == "pchip":
        f = PchipInterpolator(x, y, extrapolate=False)
        return lambda q: np.asarray(f(np.asarray(q, dtype=float)), dtype=float)
    if name == "makima":
        f = Akima1DInterpolator(x, y, method="makima", extrapolate=False)
        return lambda q: np.asarray(f(np.asarray(q, dtype=float)), dtype=float)
    raise ValueError(name)

class InterpolatedEnvironment:
    def __init__(self, t, T, C, kind):
        self.t = np.asarray(t, dtype=float)
        self.temperature = np.asarray(T, dtype=float)
        self.moisture = np.asarray(C, dtype=float)
        self._Tf = build_interpolator(kind, self.t, self.temperature)
        self._Cf = build_interpolator(kind, self.t, self.moisture)

    def T_inf(self, t):
        tq = np.asarray(t, dtype=float)
        if np.any(tq < self.t[0] - 1e-12) or np.any(tq > self.t[-1] + 1e-12):
            raise ValueError("Requested time outside interpolation range.")
        out = self._Tf(tq)
        if np.ndim(t) == 0:
            return float(np.asarray(out).reshape(-1)[0])
        return np.asarray(out, dtype=float)

    def C_inf(self, t):
        tq = np.asarray(t, dtype=float)
        if np.any(tq < self.t[0] - 1e-12) or np.any(tq > self.t[-1] + 1e-12):
            raise ValueError("Requested time outside interpolation range.")
        out = self._Cf(tq)
        if np.ndim(t) == 0:
            return float(np.asarray(out).reshape(-1)[0])
        return np.asarray(out, dtype=float)

def solve_fvm_with_interpolation(kind="linear", N=640, t_eval=None, params=None, config=None):
    params = params or q1.Q1Parameters()
    config = config or q1.SolverConfig()
    grid = q1.RadialGrid.build(N=N, R=params.R, L=params.L)
    env_interp = InterpolatedEnvironment(t_env, T_env, C_env, kind)

    if t_eval is None:
        t_eval = np.arange(0.0, params.t_end + 1.0, 1.0)
    else:
        t_eval = np.asarray(t_eval, dtype=float)

    sparsity = q1.tridiagonal_sparsity(N)
    y0_T = np.full(N, params.T0, dtype=float)
    y0_C = np.full(N, params.C0, dtype=float)

    heat = solve_ivp(
        fun=q1.make_heat_rhs(grid, env_interp, params),
        t_span=(0.0, params.t_end),
        y0=y0_T,
        method=config.method,
        t_eval=t_eval,
        rtol=config.rtol,
        atol=config.atol,
        max_step=config.max_step,
        jac_sparsity=sparsity,
    )
    if not heat.success:
        raise RuntimeError(f"Heat solve failed: {heat.message}")

    moisture = solve_ivp(
        fun=q1.make_moisture_rhs(grid, env_interp, params),
        t_span=(0.0, params.t_end),
        y0=y0_C,
        method=config.method,
        t_eval=t_eval,
        rtol=config.rtol,
        atol=config.atol,
        max_step=config.max_step,
        jac_sparsity=sparsity,
    )
    if not moisture.success:
        raise RuntimeError(f"Moisture solve failed: {moisture.message}")

    sol = q1.Q1Solution(
        params=params,
        config=config,
        grid=grid,
        env=env_interp,
        t=t_eval,
        T=heat.y.T,
        C=moisture.y.T,
        heat_solver_stats={"nfev": heat.nfev, "njev": heat.njev, "nlu": heat.nlu, "message": heat.message},
        moisture_solver_stats={"nfev": moisture.nfev, "njev": moisture.njev, "nlu": moisture.nlu, "message": moisture.message},
    )
    return sol

def solve_fdm(kind="linear", N=1280, t_eval=None, params=None, config=None):
    params = params or q1.Q1Parameters()
    config = config or q1.SolverConfig()
    env_interp = InterpolatedEnvironment(t_env, T_env, C_env, kind)

    if t_eval is None:
        t_eval = np.arange(0.0, params.t_end + 1.0, 1.0)
    else:
        t_eval = np.asarray(t_eval, dtype=float)

    dr = params.R / N
    r = np.arange(N + 1, dtype=float) * dr
    alpha = params.alpha

    jac_sp = np.zeros((N + 1, N + 1), dtype=bool)
    idx = np.arange(N + 1)
    jac_sp[idx, idx] = True
    jac_sp[idx[1:], idx[:-1]] = True
    jac_sp[idx[:-1], idx[1:]] = True

    def rhs_T(t, T):
        dT = np.empty_like(T)
        dT[0] = 4.0 * alpha * (T[1] - T[0]) / dr**2
        for j in range(1, N):
            d2 = (T[j + 1] - 2.0 * T[j] + T[j - 1]) / dr**2
            d1 = (T[j + 1] - T[j - 1]) / (2.0 * dr)
            dT[j] = alpha * (d2 + d1 / r[j])
        Tgp = T[N - 1] - 2.0 * dr * params.h / params.k * (T[N] - env_interp.T_inf(t))
        d2 = (Tgp - 2.0 * T[N] + T[N - 1]) / dr**2
        d1 = (Tgp - T[N - 1]) / (2.0 * dr)
        dT[N] = alpha * (d2 + d1 / params.R)
        return dT

    def rhs_C(t, C):
        dC = np.empty_like(C)
        Dn = q1.moisture_diffusivity(C, params)
        D12 = 2.0 * Dn[0] * Dn[1] / (Dn[0] + Dn[1])
        dC[0] = 4.0 * D12 * (C[1] - C[0]) / dr**2

        for j in range(1, N):
            rjp = r[j] + 0.5 * dr
            rjm = r[j] - 0.5 * dr
            Dp = 2.0 * Dn[j] * Dn[j + 1] / (Dn[j] + Dn[j + 1])
            Dm = 2.0 * Dn[j] * Dn[j - 1] / (Dn[j] + Dn[j - 1])
            flux_p = rjp * Dp * (C[j + 1] - C[j]) / dr
            flux_m = rjm * Dm * (C[j] - C[j - 1]) / dr
            dC[j] = (flux_p - flux_m) / (r[j] * dr)

        c_inf = env_interp.C_inf(t)
        Ds = float(q1.moisture_diffusivity(np.array([C[N]]), params)[0])
        Cgp = C[N - 1] - 2.0 * dr * params.hm / max(Ds, 1e-20) * (C[N] - c_inf)
        Dm = 2.0 * Dn[N] * Dn[N - 1] / (Dn[N] + Dn[N - 1])
        Dp = float(q1.moisture_diffusivity(np.array([(C[N] + Cgp) / 2.0]), params)[0])
        flux_p = (params.R + 0.5 * dr) * Dp * (Cgp - C[N]) / dr
        flux_m = (params.R - 0.5 * dr) * Dm * (C[N] - C[N - 1]) / dr
        dC[N] = (flux_p - flux_m) / (params.R * dr)
        return dC

    sol_T = solve_ivp(
        rhs_T, (0.0, params.t_end), np.full(N + 1, params.T0),
        method=config.method, t_eval=t_eval,
        rtol=config.rtol, atol=config.atol, max_step=config.max_step,
        jac_sparsity=jac_sp
    )
    sol_C = solve_ivp(
        rhs_C, (0.0, params.t_end), np.full(N + 1, params.C0),
        method=config.method, t_eval=t_eval,
        rtol=config.rtol, atol=config.atol, max_step=config.max_step,
        jac_sparsity=jac_sp
    )

    if not sol_T.success or not sol_C.success:
        raise RuntimeError(f"FDM solve failed: {sol_T.message}; {sol_C.message}")

    return {
        "method": "FDM",
        "interp": kind,
        "N": N,
        "dr": dr,
        "r_node": r,
        "t": sol_T.t,
        "T_node": sol_T.y.T,
        "C_node": sol_C.y.T,
    }

def fdm_to_result_grid(sol_fdm, params=None):
    params = params or q1.Q1Parameters()
    times = np.arange(0.0, params.t_end + 1.0, 1.0)
    radii_cm = np.arange(0.0, 2.0 + 0.1, 0.1)
    radii_m = radii_cm / 100.0

    Tout = np.empty((len(times), len(radii_m)))
    Cout = np.empty_like(Tout)

    for i, _ in enumerate(times):
        Tout[i] = np.interp(radii_m, sol_fdm["r_node"], sol_fdm["T_node"][i])
        Cout[i] = np.interp(radii_m, sol_fdm["r_node"], sol_fdm["C_node"][i])

    return times, radii_cm, Tout, Cout

def error_metrics(arr, ref):
    diff = np.asarray(arr) - np.asarray(ref)
    return {
        "max_abs": float(np.max(np.abs(diff))),
        "rms": float(np.sqrt(np.mean(diff**2))),
    }


## 5. Interpolation comparison table: Linear vs PCHIP vs MAKIMA

In [8]:

interp_names = ["linear", "pchip", "makima"]

# Solve and save
for name in interp_names:
    sol = solve_fvm_with_interpolation(kind=name, N=640, params=p, config=cfg)
    t_save = np.arange(0.0, p.t_end + 1.0, 1.0)
    r_save_cm = np.arange(0.0, 2.0 + 0.1, 0.1)
    r_save_m = r_save_cm / 100.0
    T_save = q1.sample_solution(sol, "T", t_save, r_save_m)
    C_save = q1.sample_solution(sol, "C", t_save, r_save_m)

    np.savez_compressed(
        NUM / f"interp_solution_{name}_N640.npz",
        time_s=t_save,
        radius_cm=r_save_cm,
        temperature_C=T_save,
        moisture_kgkg=C_save
    )

# Reload
interp_data = {
    name: dict(np.load(NUM / f"interp_solution_{name}_N640.npz"))
    for name in interp_names
}

baseline = interp_data["linear"]
rows = [{
    "Interpolation": "Linear",
    "Reference": "self",
    "E_T_max (°C)": 0.0,
    "E_T_rms (°C)": 0.0,
    "E_C_max (kg/kg)": 0.0,
    "E_C_rms (kg/kg)": 0.0,
}]
for name in ["pchip", "makima"]:
    Tm = error_metrics(interp_data[name]["temperature_C"], baseline["temperature_C"])
    Cm = error_metrics(interp_data[name]["moisture_kgkg"], baseline["moisture_kgkg"])
    rows.append({
        "Interpolation": name.capitalize(),
        "Reference": "Linear",
        "E_T_max (°C)": Tm["max_abs"],
        "E_T_rms (°C)": Tm["rms"],
        "E_C_max (kg/kg)": Cm["max_abs"],
        "E_C_rms (kg/kg)": Cm["rms"],
    })

interp_compare_df = pd.DataFrame(rows)
interp_compare_df.to_csv(TAB / "interpolation_method_error_comparison.csv", index=False, encoding="utf-8-sig")
interp_compare_df


,Interpolation,Reference,E_T_max (°C),E_T_rms (°C),E_C_max (kg/kg),E_C_rms (kg/kg)
0,Linear,self,0.000000,0.000000,0.000000,0.000000e+00
1,Pchip,Linear,0.004202,0.000629,0.000002,1.362685e-07
2,Makima,Linear,0.004338,0.000573,0.000001,1.289783e-07


**Interpretation note:**  
If the PCHIP and MAKIMA solution differences relative to Linear remain very small, then **piecewise linear interpolation** is sufficient and preferable due to simplicity, monotonicity, and reproducibility.


## 6. Generate `result1.xlsx` using the Q1 algorithm with N = 1280

In [9]:

# Main Q1 solution using the original uploaded solver and the original algorithm
sol_main = q1.solve_q1(ENV_PATH, N=1280, params=p, config=cfg)

# Save result1.xlsx
result1_path = OUT / "result1.xlsx"
q1.write_result1(sol_main, result1_path)

# Generate Table 1 and Table 2
table1_df, table2_df = q1.paper_tables(sol_main)
table1_df.to_csv(TAB / "table1_temperature.csv", encoding="utf-8-sig")
table2_df.to_csv(TAB / "table2_moisture.csv", encoding="utf-8-sig")

# Save a summary workbook
with pd.ExcelWriter(TAB / "Q1_summary_tables.xlsx", engine="openpyxl") as writer:
    interp_compare_df.to_excel(writer, sheet_name="Interp_compare", index=False)
    table1_df.to_excel(writer, sheet_name="Table1_temperature")
    table2_df.to_excel(writer, sheet_name="Table2_moisture")

print("Saved result1.xlsx:", result1_path)
print("Saved Table 1 and Table 2 CSV files.")
table1_df


Saved result1.xlsx: /fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/result1.xlsx
Saved Table 1 and Table 2 CSV files.


,0 cm,0.5 cm,1 cm,1.5 cm,2 cm
time/s,,,,,
100,28.0001,28.0003,28.0040,28.0327,28.1801
300,28.0408,28.0635,28.1514,28.3681,28.8489
600,28.4533,28.5360,28.8039,29.3159,30.1652
900,29.3243,29.4583,29.8755,30.6161,31.7304
1200,30.5427,30.7098,31.2223,32.1126,33.4276
1500,31.9957,32.1867,32.7660,33.7463,35.1203
1800,33.5753,33.7720,34.3642,35.3621,36.7856


## 7. Save main-solution grids and plot temperature/moisture heatmaps

In [10]:

times, radii_cm, T_grid, C_grid = q1.result_grid(sol_main)

np.savez_compressed(
    NUM / "main_solution_N1280.npz",
    time_s=times,
    radius_cm=radii_cm,
    temperature_C=T_grid,
    moisture_kgkg=C_grid
)

# Reload for plotting
main_data = dict(np.load(NUM / "main_solution_N1280.npz"))
times = main_data["time_s"]
radii_cm = main_data["radius_cm"]
T_grid = main_data["temperature_C"]
C_grid = main_data["moisture_kgkg"]

extent = [radii_cm[0], radii_cm[-1], times[-1], times[0]]

fig, ax = plt.subplots(figsize=(5.8, 4.2))
im = ax.imshow(T_grid, aspect="auto", extent=extent)
ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Time (s)")
ax.set_title("Temperature Heatmap")
clean_axis(ax)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Temperature (°C)")
save_dual(fig, "Fig3_temperature_heatmap")

fig, ax = plt.subplots(figsize=(5.8, 4.2))
im = ax.imshow(C_grid, aspect="auto", extent=extent)
ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Time (s)")
ax.set_title("Moisture Heatmap")
clean_axis(ax)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Moisture Concentration (kg/kg)")
save_dual(fig, "Fig4_moisture_heatmap")

print("Saved heatmaps.")


Saved heatmaps.


## 8. Plot temperature/moisture profiles at selected times

In [11]:

selected_times = np.array([100, 300, 600, 900, 1200, 1500, 1800], dtype=int)
idx_map = {int(t): int(np.where(times == t)[0][0]) for t in selected_times}

fig, ax = plt.subplots(figsize=(5.4, 3.8))
for t in selected_times:
    ax.plot(radii_cm, T_grid[idx_map[int(t)]], label=f"{t} s")
ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Temperature Profiles at Selected Times")
clean_axis(ax)
ax.legend(frameon=False, ncol=2)
save_dual(fig, "Fig5_temperature_profiles_selected_times")

fig, ax = plt.subplots(figsize=(5.4, 3.8))
for t in selected_times:
    ax.plot(radii_cm, C_grid[idx_map[int(t)]], label=f"{t} s")
ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Moisture Concentration (kg/kg)")
ax.set_title("Moisture Profiles at Selected Times")
clean_axis(ax)
ax.legend(frameon=False, ncol=2)
save_dual(fig, "Fig6_moisture_profiles_selected_times")

print("Saved selected-time profiles.")


Saved selected-time profiles.


## 9. FVM vs FDM: save results, error table, and comparison figures

In [12]:

# Solve FDM on the same time horizon and grid resolution
sol_fdm = solve_fdm(kind="linear", N=1280, params=p, config=cfg)
t_fdm, r_fdm_cm, T_fdm, C_fdm = fdm_to_result_grid(sol_fdm, params=p)

np.savez_compressed(
    NUM / "fdm_solution_N1280.npz",
    time_s=t_fdm,
    radius_cm=r_fdm_cm,
    temperature_C=T_fdm,
    moisture_kgkg=C_fdm
)

fdm_data = dict(np.load(NUM / "fdm_solution_N1280.npz"))

# Error table against the main FVM solution
temp_err = error_metrics(fdm_data["temperature_C"], T_grid)
moist_err = error_metrics(fdm_data["moisture_kgkg"], C_grid)

fvm_fdm_error_df = pd.DataFrame([
    {
        "Method A": "FVM (our algorithm)",
        "Method B": "FDM",
        "Field": "Temperature",
        "Max abs error": temp_err["max_abs"],
        "RMS error": temp_err["rms"],
        "Unit": "°C",
    },
    {
        "Method A": "FVM (our algorithm)",
        "Method B": "FDM",
        "Field": "Moisture",
        "Max abs error": moist_err["max_abs"],
        "RMS error": moist_err["rms"],
        "Unit": "kg/kg",
    },
])

fvm_fdm_error_df.to_csv(TAB / "fvm_vs_fdm_error_analysis.csv", index=False, encoding="utf-8-sig")
fvm_fdm_error_df


,Method A,Method B,Field,Max abs error,RMS error,Unit
0,FVM (our algorithm),FDM,Temperature,0.000002,5.837248e-07,°C
1,FVM (our algorithm),FDM,Moisture,0.000101,2.654768e-06,kg/kg


In [13]:

# Error histories over time
temp_diff = np.abs(fdm_data["temperature_C"] - T_grid)
moist_diff = np.abs(fdm_data["moisture_kgkg"] - C_grid)

temp_max_time = temp_diff.max(axis=1)
moist_max_time = moist_diff.max(axis=1)

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.plot(times, temp_max_time)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Max absolute error (°C)")
ax.set_title("FVM vs FDM: Temperature Error History")
clean_axis(ax)
save_dual(fig, "Fig7_fvm_fdm_temperature_error_history")

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.plot(times, moist_max_time)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Max absolute error (kg/kg)")
ax.set_title("FVM vs FDM: Moisture Error History")
clean_axis(ax)
save_dual(fig, "Fig8_fvm_fdm_moisture_error_history")

print("Saved error-history figures.")


Saved error-history figures.


In [14]:

# Final-time profile comparison
fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.plot(radii_cm, T_grid[-1], label="FVM")
ax.plot(fdm_data["radius_cm"], fdm_data["temperature_C"][-1], label="FDM")
ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Final Temperature Profile at 1800 s")
clean_axis(ax)
ax.legend(frameon=False)
save_dual(fig, "Fig9_fvm_fdm_final_temperature_profile")

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.plot(radii_cm, C_grid[-1], label="FVM")
ax.plot(fdm_data["radius_cm"], fdm_data["moisture_kgkg"][-1], label="FDM")
ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Moisture Concentration (kg/kg)")
ax.set_title("Final Moisture Profile at 1800 s")
clean_axis(ax)
ax.legend(frameon=False)
save_dual(fig, "Fig10_fvm_fdm_final_moisture_profile")

print("Saved final-time comparison figures.")


Saved final-time comparison figures.


# 网格分析

In [17]:
# ============================================================
# Grid convergence analysis
# N = 160, 320, 640, 1280
# Reference solution: N = 2560
# Error evaluated on the required 1 s × 0.1 cm output grid
# ============================================================

print("=" * 70)
print("Grid convergence analysis")
print("=" * 70)

# ------------------------------------------------------------
# 1. Grid settings
# ------------------------------------------------------------
grid_N_list = [160, 320, 640, 1280]
reference_N = 2560

# Required output grid of Question 1
grid_times = np.arange(0.0, 1800.0 + 1.0, 1.0)
grid_radii_cm = np.arange(0.0, 2.0 + 0.1, 0.1)
grid_radii_m = grid_radii_cm / 100.0


# ------------------------------------------------------------
# 2. Solve different grids
# ------------------------------------------------------------
grid_solutions = {}

for N in grid_N_list:
    print(f"Solving FVM: N = {N} ...")

    # Reuse the already-computed production solution if available
    if N == 1280 and "sol_main" in globals():
        sol_N = sol_main
        print("  -> reused existing sol_main")
    else:
        sol_N = q1.solve_q1(
            ENV_PATH,
            N=N,
            params=p,
            config=cfg
        )

    grid_solutions[N] = sol_N


# ------------------------------------------------------------
# 3. Fine-grid reference solution
# ------------------------------------------------------------
print(f"Solving reference solution: N = {reference_N} ...")

sol_ref_grid = q1.solve_q1(
    ENV_PATH,
    N=reference_N,
    params=p,
    config=cfg
)

print("Reference solution completed.")


# ------------------------------------------------------------
# 4. Sample all solutions on exactly the same output grid
# ------------------------------------------------------------
T_ref_grid = q1.sample_solution(
    sol_ref_grid,
    "T",
    grid_times,
    grid_radii_m
)

C_ref_grid = q1.sample_solution(
    sol_ref_grid,
    "C",
    grid_times,
    grid_radii_m
)


grid_records = []

for N in grid_N_list:

    sol_N = grid_solutions[N]

    T_N = q1.sample_solution(
        sol_N,
        "T",
        grid_times,
        grid_radii_m
    )

    C_N = q1.sample_solution(
        sol_N,
        "C",
        grid_times,
        grid_radii_m
    )

    # --------------------------------------------------------
    # Error relative to fine reference N = 2560
    # --------------------------------------------------------
    err_T = T_N - T_ref_grid
    err_C = C_N - C_ref_grid

    E_T_max = np.max(np.abs(err_T))
    E_T_rms = np.sqrt(np.mean(err_T**2))

    E_C_max = np.max(np.abs(err_C))
    E_C_rms = np.sqrt(np.mean(err_C**2))

    grid_records.append({
        "N": N,
        "dr (cm)": p.R * 100.0 / N,

        "E_T_max (°C)": E_T_max,
        "E_T_rms (°C)": E_T_rms,

        "E_C_max (kg/kg)": E_C_max,
        "E_C_rms (kg/kg)": E_C_rms,
    })

    # Save each numerical result for later inspection
    np.savez_compressed(
        NUM / f"grid_solution_N{N}.npz",
        time_s=grid_times,
        radius_cm=grid_radii_cm,
        temperature_C=T_N,
        moisture_kgkg=C_N,
    )


# ------------------------------------------------------------
# 5. Construct convergence table
# ------------------------------------------------------------
grid_analysis_df = pd.DataFrame(grid_records)

grid_analysis_df["Observed order T"] = np.nan
grid_analysis_df["Observed order C"] = np.nan

# p = log(E_h / E_h/2) / log(2)
for i in range(len(grid_analysis_df) - 1):

    E_T_1 = grid_analysis_df.loc[i, "E_T_max (°C)"]
    E_T_2 = grid_analysis_df.loc[i + 1, "E_T_max (°C)"]

    E_C_1 = grid_analysis_df.loc[i, "E_C_max (kg/kg)"]
    E_C_2 = grid_analysis_df.loc[i + 1, "E_C_max (kg/kg)"]

    if E_T_1 > 0 and E_T_2 > 0:
        grid_analysis_df.loc[i, "Observed order T"] = (
            np.log(E_T_1 / E_T_2) / np.log(2.0)
        )

    if E_C_1 > 0 and E_C_2 > 0:
        grid_analysis_df.loc[i, "Observed order C"] = (
            np.log(E_C_1 / E_C_2) / np.log(2.0)
        )


# ------------------------------------------------------------
# 6. Save convergence table
# ------------------------------------------------------------
grid_table_path = TAB / "grid_convergence_analysis.csv"

grid_analysis_df.to_csv(
    grid_table_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nGrid convergence table:")
display(grid_analysis_df)

print("\nSaved:")
print(grid_table_path)


# ------------------------------------------------------------
# 7. Temperature grid-convergence figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.6))

ax.loglog(
    grid_analysis_df["N"],
    grid_analysis_df["E_T_max (°C)"],
    marker="o",
    label="Maximum error"
)

ax.loglog(
    grid_analysis_df["N"],
    grid_analysis_df["E_T_rms (°C)"],
    marker="s",
    linestyle="--",
    label="RMS error"
)

ax.set_xlabel("Number of radial cells, N")
ax.set_ylabel("Temperature Error (°C)")
ax.set_title("Grid Convergence of Temperature")

clean_axis(ax)
ax.legend(frameon=False)

save_dual(
    fig,
    "Fig_grid_convergence_temperature"
)


# ------------------------------------------------------------
# 8. Moisture grid-convergence figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.6))

ax.loglog(
    grid_analysis_df["N"],
    grid_analysis_df["E_C_max (kg/kg)"],
    marker="o",
    label="Maximum error"
)

ax.loglog(
    grid_analysis_df["N"],
    grid_analysis_df["E_C_rms (kg/kg)"],
    marker="s",
    linestyle="--",
    label="RMS error"
)

ax.set_xlabel("Number of radial cells, N")
ax.set_ylabel("Moisture Error (kg/kg)")
ax.set_title("Grid Convergence of Moisture")

clean_axis(ax)
ax.legend(frameon=False)

save_dual(
    fig,
    "Fig_grid_convergence_moisture"
)


# ------------------------------------------------------------
# 9. Compare radial profiles at t = 1800 s
# ------------------------------------------------------------
profile_time = 1800.0

fig, ax = plt.subplots(figsize=(5.2, 3.6))

for N in grid_N_list:

    T_profile = q1.sample_solution(
        grid_solutions[N],
        "T",
        [profile_time],
        grid_radii_m
    )[0]

    ax.plot(
        grid_radii_cm,
        T_profile,
        label=f"N = {N}"
    )

# Reference
T_ref_profile = q1.sample_solution(
    sol_ref_grid,
    "T",
    [profile_time],
    grid_radii_m
)[0]

ax.plot(
    grid_radii_cm,
    T_ref_profile,
    linestyle="--",
    label=f"Reference N = {reference_N}"
)

ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Temperature Profiles at 1800 s")

clean_axis(ax)
ax.legend(frameon=False, ncol=2)

save_dual(
    fig,
    "Fig_grid_temperature_profiles_1800s"
)


# ------------------------------------------------------------
# 10. Moisture profiles at t = 1800 s
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.6))

for N in grid_N_list:

    C_profile = q1.sample_solution(
        grid_solutions[N],
        "C",
        [profile_time],
        grid_radii_m
    )[0]

    ax.plot(
        grid_radii_cm,
        C_profile,
        label=f"N = {N}"
    )

# Reference
C_ref_profile = q1.sample_solution(
    sol_ref_grid,
    "C",
    [profile_time],
    grid_radii_m
)[0]

ax.plot(
    grid_radii_cm,
    C_ref_profile,
    linestyle="--",
    label=f"Reference N = {reference_N}"
)

ax.set_xlabel("Radius (cm)")
ax.set_ylabel("Moisture Concentration (kg/kg)")
ax.set_title("Moisture Profiles at 1800 s")

clean_axis(ax)
ax.legend(frameon=False, ncol=2)

save_dual(
    fig,
    "Fig_grid_moisture_profiles_1800s"
)


# ------------------------------------------------------------
# 11. Save reference solution
# ------------------------------------------------------------
np.savez_compressed(
    NUM / f"grid_reference_N{reference_N}.npz",
    time_s=grid_times,
    radius_cm=grid_radii_cm,
    temperature_C=T_ref_grid,
    moisture_kgkg=C_ref_grid,
)


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("Grid convergence analysis completed")
print("=" * 70)

print(f"Reference grid : N = {reference_N}")
print(f"Production grid: N = 1280")

print("\nOutput table:")
print(TAB / "grid_convergence_analysis.csv")

print("\nOutput figures:")
print(FIG / "Fig_grid_convergence_temperature.png")
print(FIG / "Fig_grid_convergence_temperature.svg")

print(FIG / "Fig_grid_convergence_moisture.png")
print(FIG / "Fig_grid_convergence_moisture.svg")

print(FIG / "Fig_grid_temperature_profiles_1800s.png")
print(FIG / "Fig_grid_temperature_profiles_1800s.svg")

print(FIG / "Fig_grid_moisture_profiles_1800s.png")
print(FIG / "Fig_grid_moisture_profiles_1800s.svg")


Grid convergence analysis
Solving FVM: N = 160 ...
Solving FVM: N = 320 ...
Solving FVM: N = 640 ...
Solving FVM: N = 1280 ...
  -> reused existing sol_main
Solving reference solution: N = 2560 ...
Reference solution completed.

Grid convergence table:


,N,dr (cm),E_T_max (°C),E_T_rms (°C),E_C_max (kg/kg),E_C_rms (kg/kg),Observed order T,Observed order C
0,160,0.012500,0.000055,3.969352e-05,0.002264,7.681092e-05,2.014750,4.090503
1,320,0.006250,0.000014,9.815360e-06,0.000133,1.879799e-05,2.028667,2.083195
2,640,0.003125,0.000003,2.364606e-06,0.000031,4.480873e-06,1.517065,2.320777
3,1280,0.001563,0.000001,4.881606e-07,0.000006,8.969542e-07,NaN,NaN



Saved:
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/tables/grid_convergence_analysis.csv

Grid convergence analysis completed
Reference grid : N = 2560
Production grid: N = 1280

Output table:
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/tables/grid_convergence_analysis.csv

Output figures:
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/figures/Fig_grid_convergence_temperature.png
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/figures/Fig_grid_convergence_temperature.svg
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/figures/Fig_grid_convergence_moisture.png
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/figures/Fig_grid_convergence_moisture.svg
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/figures/Fig_grid_temperature_profiles_1800s.png
/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATA

## 10. Save a run summary

In [15]:

summary = {
    "result1_path": str(result1_path),
    "figure_count_png": len(list(FIG.glob("*.png"))),
    "figure_count_svg": len(list(FIG.glob("*.svg"))),
    "table_count_csv": len(list(TAB.glob("*.csv"))),
    "saved_npz": len(list(NUM.glob("*.npz"))),
}
with open(OUT / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary


{'result1_path': '/fs/ess/PAS1475/wangxiaoying/YYhao/Rare_ATAC/MATH_MODEL/Q1_targeted_outputs/result1.xlsx',
 'figure_count_png': 10,
 'figure_count_svg': 10,
 'table_count_csv': 4,
 'saved_npz': 5}

## Recommended use in the paper

- **Fig. 1–2**: justify the preprocessing of ambient input data.
- **Interpolation comparison table**: explain why **piecewise linear interpolation** is adequate.
- **Table 1 and Table 2**: directly support the required Q1 result reporting.
- **Heatmaps**: show the global time–radius evolution of temperature and moisture.
- **Selected-time radial profiles**: reveal the spatial propagation process clearly.
- **FVM vs FDM error table and figures**: support the numerical-method discussion and justify the choice of the main algorithm.
